# HRV exploratory analysis: sliding-window design (v3)

Exploratory analysis of heart rate variability for the Mentasti design adapted to MIMIC-III: T0 every 6 h, 12-h observation window, 6-h horizon.

The data exist at two levels:

| Level | File | Unit | Question |
|---|---|---|---|
| **A. Hour** | `hrv_completo_v3.csv` | 1 row = 1 hour with valid HRV | Quality and coverage of the extracted signal |
| **B. Window** | `ultimate_dataset_v3_1.csv` | 1 row = 1 window (patient, T0) | What the model receives |

**Rules**
1. Descriptive only: no imputation, outlier removal or feature selection here. All of that happens inside cross-validation, on the training fold.
2. The >= 6 h valid-HRV threshold was fixed a priori; nothing seen here moves it.
3. Windows of the same patient are not independent, so group comparisons use the **standardised mean difference (SMD)**, not p-values.

In [ ]:
# 0. Setup and data loading
import os, json, math, warnings
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
warnings.filterwarnings('ignore')
sns.set_style('whitegrid')
pd.set_option('display.max_columns', None)

try:
    from google.colab import drive
    drive.mount('/content/drive')
    EN_COLAB = True
except ImportError:
    EN_COLAB = False

try:
    display
except NameError:
    display = print

RUTA_BASE = '/content/drive/MyDrive/tesis_sepsis'
RUTA_HRV = f'{RUTA_BASE}/hrv_completo_v3.csv'
RUTA_ULTIMATE = f'{RUTA_BASE}/ultimate_dataset_v3_1.csv'
RUTA_COLUMNAS = f'{RUTA_BASE}/columnas_v3_1.json'
RUTA_REPORTE = f'{RUTA_BASE}/df_reporte_final_unificado.csv'
RUTA_EDA = f'{RUTA_BASE}/eda_hrv_v3_1'  # tables and figures of this EDA
os.makedirs(RUTA_EDA, exist_ok=True)

METRICAS = ['SDNN', 'RMSSD', 'LF', 'HF', 'LFHF', 'SD1', 'SD2', 'SampEn', 'ApEn', 'DFA_a1']
DOMINIOS = {'Temporal': ['SDNN', 'RMSSD', 'SD1', 'SD2'],  # SD1 and SD2: linear (Brennan et al., 2001)
            'Frecuencial': ['LF', 'HF', 'LFHF'],
            'No lineal': ['SampEn', 'ApEn', 'DFA_a1']}
COLS_T0 = [f'{m}_T0' for m in METRICAS]
UMBRAL_HRV = 6  # fixed a priori

LABELS = {
    'label_sofa_responder':      ('SOFA respuesta (principal)',  {1: 'Responder', 0: 'No responder'}),
    'label_cardio':              ('Cardiovascular (principal)',  {1: 'Responder', 0: 'No responder'}),
    'label_sofa_deterioro_2pts': ('SOFA deterioro ≥2 (secundario)', {1: 'Deterioro', 0: 'Sin deterioro'}),
}

# --- Hour level ---
def valor_horario_mediana(df):
    """Replace {m}_T0 by the median of the valid 5-min epochs of the hour
    (last epoch = {m}_T0; epoch N min earlier = {m}_T0 - Delta_{m}_minus_{N}min)."""
    out = df.copy()
    for m in METRICAS:
        ancla = df[f'{m}_T0']
        epocas = [ancla - df[f'Delta_{m}_minus_{k}min'] for k in range(5, 60, 5)]
        epocas.append(ancla)
        out[f'{m}_T0'] = pd.concat(epocas, axis=1).median(axis=1, skipna=True)
    return out

hrv_ep12 = pd.read_csv(RUTA_HRV)  # epoch-12 value (only for check A5)
for c in ['icustay_id', 'subject_id', 'hour_index']:
    hrv_ep12[c] = hrv_ep12[c].astype('int64')
hrv = valor_horario_mediana(hrv_ep12)  # hourly value = median of the valid epochs

# --- Window level ---
ult = pd.read_csv(RUTA_ULTIMATE, parse_dates=['t0_prediccion'])
with open(RUTA_COLUMNAS) as f:
    COLS = json.load(f)
FEAT_HRV = COLS['features_hrv']  # 40 + horas_desde_ultima_hrv
FEAT_HRV_40 = [c for c in FEAT_HRV if c != 'horas_desde_ultima_hrv']

# Analytic cohort: windows with >= 6 h of valid HRV
dfv = ult[ult[f'incl_hrv_{UMBRAL_HRV}h'] == 1].copy()

# --- Synchronisation metadata (optional: tipo_t0, horas_disponibles) ---
reporte = pd.read_csv(RUTA_REPORTE) if os.path.exists(RUTA_REPORTE) else None

print(f"HRV horario          : {hrv.shape}, {hrv.icustay_id.nunique()} pacientes")
print(f"Ventanas candidatas  : {len(ult)}, {ult.icustay_id.nunique()} pacientes")
print(f"Ventanas ≥{UMBRAL_HRV}h HRV     : {len(dfv)}, {dfv.icustay_id.nunique()} pacientes")
print(f"Reporte sincronización: {'cargado ' + str(reporte.shape) if reporte is not None else 'no encontrado'}")

# Scale of LF, HF and LFHF: a_ln applies ln only if all values are > 0 (original scale);
# if there are negative values (already ln) it does nothing.
def a_ln(v):
    v = pd.Series(v, dtype=float)
    if (v.dropna() > 0).all():
        return np.log(v)
    return v

for _m in ['LF', 'HF', 'LFHF']:
    _v = ult[f'{_m}_mean'].dropna() if f'{_m}_mean' in ult else pd.Series(dtype=float)
    print(f"{_m}_mean en ultimate_dataset: min {_v.min():.4g} → {'escala ORIGINAL (se aplicará ln en figuras)' if (_v > 0).all() else 'ya en ln'}")


---
# Part A. Hour level: the extracted signal

## A1. Structure

Each row is one hour with good-quality HRV (ectopic load < 5 % and >= 8/11 valid 5-min epochs). Hours without good signal are absent, so rows per patient = valid hours.

In [ ]:
print("=" * 70); print("A1. ESTRUCTURA DEL HRV HORARIO"); print("=" * 70)
display(hrv[['subject_id', 'icustay_id', 'hour_index'] + COLS_T0].head())

g = hrv.groupby('icustay_id')
filas = g.size()
rango = g['hour_index'].max() - g['hour_index'].min() + 1
huecos = rango - filas

print(f"\nDuplicados (icustay_id, hour_index): {hrv.duplicated(['icustay_id','hour_index']).sum()}")
print(f"Filas == horas_validas_total: {(filas == g['horas_validas_total'].first()).mean():.3f}")
print(f"hour_index: {hrv.hour_index.min()} a {hrv.hour_index.max()}")

resumen_pac = pd.DataFrame({'horas_validas': filas, 'rango_horas': rango, 'horas_faltantes_en_rango': huecos})
display(resumen_pac.describe().round(1))

fig, axes = plt.subplots(1, 2, figsize=(14, 4))
sns.histplot(filas, bins=30, ax=axes[0], color='teal')
axes[0].axvline(12, ls='--', c='k'); axes[0].text(13, axes[0].get_ylim()[1]*0.9, '12 h (mínimo para 1 ventana)')
axes[0].set(title='Horas válidas por paciente', xlabel='Horas', ylabel='Pacientes')
sns.histplot(huecos, bins=30, ax=axes[1], color='slategray')
axes[1].set(title='Horas faltantes dentro del rango de cada paciente', xlabel='Horas', ylabel='Pacientes')
plt.tight_layout(); plt.savefig(f'{RUTA_EDA}/A1_horas_por_paciente.png', dpi=150); plt.show()

print(f"\n>> Pacientes sin horas válidas desde la hora 11 (no generan ninguna ventana): {(g['hour_index'].max() < 11).sum()} de {len(filas)}")

## A2. Temporal coverage

How many patients contribute HRV at each hour from the HRV anchor? If coverage drops quickly, late windows represent a subgroup (usually sicker patients who remain monitored).

In [ ]:
print("=" * 70); print("A2. COBERTURA TEMPORAL"); print("=" * 70)
n_pac = hrv.icustay_id.nunique()
por_hora = hrv.groupby('hour_index')['icustay_id'].nunique().reindex(range(96), fill_value=0)

fig, axes = plt.subplots(1, 2, figsize=(15, 4.5), gridspec_kw={'width_ratios': [1, 1.3]})
axes[0].plot(por_hora.index, por_hora / n_pac * 100, color='teal')
for t in range(12, 97, 6):
    axes[0].axvline(t, color='orange', alpha=0.25, lw=0.8)
axes[0].set(title='Pacientes con HRV válida por hora (líneas: grilla de T0)', xlabel='Horas desde el inicio', ylabel='% de pacientes')

# Patient x hour map (sorted by valid hours)
orden = filas.sort_values(ascending=False).index
mat = (hrv.assign(v=1).pivot_table(index='icustay_id', columns='hour_index', values='v', fill_value=0)
          .reindex(index=orden, columns=range(96), fill_value=0))
axes[1].imshow(mat.values, aspect='auto', cmap='Greys', interpolation='nearest')
axes[1].set(title='Mapa de señal válida (fila = paciente, negro = hora válida)', xlabel='Horas desde el inicio', ylabel='Pacientes (ordenados)')
axes[1].set_yticks([]); axes[1].grid(False)
plt.tight_layout(); plt.savefig(f'{RUTA_EDA}/A2_cobertura_temporal.png', dpi=150); plt.show()

display(pd.DataFrame({'hora': [0, 12, 24, 48, 72, 95],
                      '% pacientes': [round(por_hora[h] / n_pac * 100, 1) for h in [0, 12, 24, 48, 72, 95]]}))

## A3. Distribution of the 10 hourly metrics

Spectral powers (LF, HF) and their ratio are right-skewed and are also shown on the log scale.

In [ ]:
print("=" * 70); print("A3. DISTRIBUCIÓN DE MÉTRICAS HORARIAS"); print("=" * 70)
desc = hrv[COLS_T0].describe().T
desc['asimetria'] = hrv[COLS_T0].skew()
display(desc.round(4))
desc.to_csv(f'{RUTA_EDA}/A3_descriptivo_horario.csv')

LOG = {'LF_T0', 'HF_T0', 'LFHF_T0'}
fig, axes = plt.subplots(4, 3, figsize=(18, 15)); axes = axes.flatten()
for i, col in enumerate(COLS_T0):
    s = hrv[col].dropna()
    if col in LOG:
        s = a_ln(s[s > 0])
        sns.histplot(s, bins=40, kde=True, ax=axes[i], color='coral')
        axes[i].set(title=f'{col} (ln)', xlabel='ln(valor)')
    else:
        sns.histplot(s, bins=40, kde=True, ax=axes[i], color='teal')
        axes[i].set(title=col, xlabel='Valor')
for j in range(i + 1, len(axes)):
    fig.delaxes(axes[j])
plt.tight_layout(); plt.savefig(f'{RUTA_EDA}/A3_distribuciones.png', dpi=150); plt.show()

## A4. Missing data

Description only. The T0 metrics should have no missing values (an hour without HRV is not a row); the within-hour deltas can, when a 5-min reference epoch is missing. The deltas are not used by the main models.

In [ ]:
print("=" * 70); print("A4. DATOS AUSENTES (HORARIO)"); print("=" * 70)
def analizar_nulos(df, cols, nombre, top_n=20, graficar=True):
    pct = (df[cols].isna().mean() * 100).sort_values(ascending=False)
    tabla = pd.DataFrame({'n_nulos': df[cols].isna().sum()[pct.index], 'pct_nulos': pct.round(2)})
    tabla = tabla[tabla.n_nulos > 0]
    print(f"\n--- {nombre}: {len(df)} filas, {len(tabla)} de {len(cols)} columnas con nulos ---")
    if tabla.empty:
        print("Sin valores nulos."); return tabla
    display(tabla.head(top_n))
    if graficar:
        top = tabla.head(top_n)
        plt.figure(figsize=(9, max(3, 0.3 * len(top))))
        sns.barplot(x=top.pct_nulos, y=top.index, color='indianred')
        plt.title(f'Columnas con más nulos — {nombre}'); plt.xlabel('% nulos'); plt.tight_layout(); plt.show()
    return tabla

cols_delta = [c for c in hrv.columns if c.startswith('Delta_')]
nulos_t0 = analizar_nulos(hrv, COLS_T0, 'Métricas T0')
nulos_delta = analizar_nulos(hrv, cols_delta, 'Deltas intra-hora', top_n=10, graficar=False)
if not nulos_delta.empty:
    por_lag = nulos_delta.assign(lag=nulos_delta.index.str.extract(r'minus_(\d+)min', expand=False).values.astype(int)) \
                         .groupby('lag')['pct_nulos'].mean()
    print("\n% medio de nulos por rezago del delta (min):"); display(por_lag.round(2))

## A5. Outliers and physiological checks

- **Statistical outlier (IQR):** far from the rest; in sepsis it may be a real extreme patient, so it is not removed.
- **Physiologically or mathematically impossible:** negative power, SDNN <= 0, inconsistent LF/HF. This would indicate an extraction error to be fixed at the source.

Also checks the identity **SD1 = RMSSD / sqrt(2)**: if it holds, both metrics carry the same information (relevant for A6).

In [ ]:
print("=" * 70); print("A5. ATÍPICOS (IQR) Y CHEQUEO FISIOLÓGICO"); print("=" * 70)
def detectar_outliers_iqr(df, columnas, nombre):
    res = []
    for c in columnas:
        s = df[c].dropna()
        if len(s) < 5: continue
        q1, q3 = s.quantile([.25, .75]); iqr = q3 - q1
        li, ls = q1 - 1.5 * iqr, q3 + 1.5 * iqr
        n = ((s < li) | (s > ls)).sum()
        res.append({'variable': c, 'n_atipicos': n, 'pct_atipicos': round(100 * n / len(s), 2),
                    'lim_inf': round(li, 4), 'lim_sup': round(ls, 4),
                    'min': round(s.min(), 4), 'max': round(s.max(), 4)})
    out = pd.DataFrame(res).sort_values('pct_atipicos', ascending=False)
    print(f"\n--- Atípicos IQR: {nombre} ---"); display(out); return out

out_hora = detectar_outliers_iqr(hrv, COLS_T0, 'HRV horario')
out_hora.to_csv(f'{RUTA_EDA}/A5_atipicos_horario.csv', index=False)

fig, axes = plt.subplots(2, 5, figsize=(20, 8)); axes = axes.flatten()
for i, c in enumerate(COLS_T0):
    sns.boxplot(y=hrv[c], ax=axes[i], color='lightsteelblue', fliersize=1.5)
    axes[i].set_title(c)
    if c in {'LF_T0', 'HF_T0', 'LFHF_T0'}: axes[i].set_yscale('log')
plt.suptitle('Boxplots HRV horario (LF, HF y LF/HF en escala log)', y=1.01); plt.tight_layout(); plt.show()


def sanity_check_hrv(df, nombre):
    n = len(df); fails, alertas, infos = [], [], []
    for m in ['SDNN', 'RMSSD', 'SD1', 'SD2']:
        k = (df[f'{m}_T0'] <= 0).sum()
        if k: fails.append(f"{m}_T0 ≤ 0 en {k} filas (desviación no positiva: imposible)")
    for m in ['LF', 'HF']:
        k = (df[f'{m}_T0'] < 0).sum()
        if k: fails.append(f"{m}_T0 < 0 en {k} filas (potencia negativa: imposible)")
    for m in ['SampEn', 'ApEn']:
        k = (df[f'{m}_T0'] < 0).sum()
        if k: fails.append(f"{m}_T0 < 0 en {k} filas (entropía negativa: imposible)")
        k = (df[f'{m}_T0'] > 5).sum()
        if k: infos.append(f"{m}_T0 > 5 en {k} filas (alto; revisar señal)")
    # coherent LF/HF
    calc = df['LF_T0'] / df['HF_T0'].replace(0, np.nan)
    k = ((df['LFHF_T0'] - calc).abs() > 0.05 * calc.abs().clip(lower=1e-6)).sum()
    if k: fails.append(f"LFHF_T0 ≠ LF/HF recalculado (>5 %) en {k} filas")
    # SD1 = RMSSD/sqrt(2) (mathematical identity)
    ratio = df['SD1_T0'] / (df['RMSSD_T0'] / np.sqrt(2))
    infos.append(f"SD1 / (RMSSD/√2): mediana {ratio.median():.3f}, P1–P99 {ratio.quantile(.01):.3f}–{ratio.quantile(.99):.3f} "
                 f"(≈1 confirma que SD1 y RMSSD son redundantes)")
    k = (df['RMSSD_T0'] > 3 * df['SDNN_T0']).sum()
    if k: alertas.append(f"RMSSD_T0 > 3×SDNN_T0 en {k} filas (relación rara; posible ectopia residual)")
    k = (df['DFA_a1_T0'] < 0).sum()
    if k: infos.append(f"DFA_a1_T0 < 0 en {k} filas (posible ruido o arritmia)")
    k = (df['SD1_T0'] > df['SD2_T0']).sum()
    infos.append(f"SD1 > SD2 en {k} filas ({100*k/n:.1f} %): plausible en sepsis, no se marca como error")
    # wide physiological ranges in ms
    for m, lim in [('SDNN', 300), ('RMSSD', 300)]:
        k = (df[f'{m}_T0'] > lim).sum()
        if k: alertas.append(f"{m}_T0 > {lim} ms en {k} filas (improbable en UCI; revisar artefacto)")

    print(f"\n--- Chequeo fisiológico: {nombre} ({n} filas) ---")
    for i in infos: print(f"  ℹ️  {i}")
    for a in alertas: print(f"  ⚠️  {a}")
    if fails:
        print(f"  ❌ {len(fails)} tipos de violaciones críticas:"); [print(f"      - {f}") for f in fails]
    else:
        print("  ✅ Sin violaciones críticas.")
    return {'fails': fails, 'alertas': alertas, 'infos': infos}

# On the final epoch: LF/HF = LF/HF and SD1 = RMSSD/sqrt(2) hold epoch by epoch, not for medians
chequeo_hora = sanity_check_hrv(hrv_ep12, 'HRV horario (época final)')

## A6. Correlation between metrics

Spearman correlation (several metrics are skewed). Pairs with |rho| >= 0.9 carry practically the same information, which matters when reading importances (SHAP splits credit among redundant variables).

In [ ]:
print("=" * 70); print("A6. CORRELACIÓN ENTRE MÉTRICAS (SPEARMAN, HORARIO)"); print("=" * 70)
def clasificar(r):
    r = abs(r)
    return 'Fuerte (≥0,7)' if r >= .7 else 'Moderada (0,4–0,7)' if r >= .4 else 'Débil (0,1–0,4)' if r >= .1 else 'Nula (<0,1)'

def tabla_correlacion(df, cols, nombre, metodo='spearman', anotar=True, tam=(10, 8)):
    corr = df[cols].corr(method=metodo)
    plt.figure(figsize=tam)
    sns.heatmap(corr, mask=np.triu(np.ones_like(corr, dtype=bool)), annot=anotar, fmt='.2f',
                cmap='coolwarm', center=0, vmin=-1, vmax=1, square=True, cbar_kws={'shrink': .7})
    plt.title(f'Correlación de {metodo.capitalize()} — {nombre}'); plt.tight_layout(); plt.show()
    pares = corr.where(np.triu(np.ones(corr.shape), k=1).astype(bool)).stack().reset_index()
    pares.columns = ['variable_1', 'variable_2', 'rho']
    pares = pares.reindex(pares.rho.abs().sort_values(ascending=False).index).reset_index(drop=True)
    pares['fuerza'] = pares.rho.apply(clasificar)
    print(f"\nTop 10 pares ({nombre}):"); display(pares.head(10).round(3))
    print("\nDistribución de fuerza:"); display(pares.fuerza.value_counts())
    redund = pares[pares.rho.abs() >= .9]
    print(f"\nPares redundantes |ρ| ≥ 0,9: {len(redund)}")
    if len(redund): display(redund.round(3))
    return corr, pares

corr_hora, pares_hora = tabla_correlacion(hrv, COLS_T0, 'HRV horario')

## A7. Stability: between-patient versus within-patient variability

**ICC(1)** estimates the share of total variability due to differences between patients. A high ICC (> 0.75) means the metric is a stable patient trait (the window mean summarises it well); a low ICC means it fluctuates hour to hour, so the window SD and slope may carry signal.

In [ ]:
print("=" * 70); print("A7. ICC(1): VARIABILIDAD ENTRE vs DENTRO DE PACIENTES"); print("=" * 70)
def icc1(df, col, grupo='icustay_id'):
    d = df[[grupo, col]].dropna()
    ni = d.groupby(grupo).size(); d = d[d[grupo].isin(ni[ni >= 2].index)]
    ni = d.groupby(grupo).size(); k = len(ni); N = ni.sum()
    if k < 3: return np.nan
    media = d[col].mean(); medias_g = d.groupby(grupo)[col].mean()
    ssb = (ni * (medias_g - media) ** 2).sum(); ssw = ((d[col] - d[grupo].map(medias_g)) ** 2).sum()
    msb, msw = ssb / (k - 1), ssw / (N - k)
    k0 = (N - (ni ** 2).sum() / N) / (k - 1)
    return (msb - msw) / (msb + (k0 - 1) * msw)

tabla_icc = pd.DataFrame({'ICC1': [icc1(hrv, c) for c in COLS_T0]}, index=COLS_T0).round(3)
tabla_icc['interpretación'] = pd.cut(tabla_icc.ICC1, [-np.inf, .5, .75, .9, np.inf],
                                     labels=['Baja: fluctúa hora a hora', 'Moderada', 'Buena', 'Excelente: rasgo estable'])
display(tabla_icc); tabla_icc.to_csv(f'{RUTA_EDA}/A7_icc.csv')

plt.figure(figsize=(9, 4.5))
sns.barplot(x=tabla_icc.index, y=tabla_icc.ICC1, color='teal')
for y, lab in [(.5, 'moderada'), (.75, 'buena'), (.9, 'excelente')]:
    plt.axhline(y, ls='--', c='gray', lw=.8); plt.text(9.6, y, lab, va='center', fontsize=9)
plt.ylim(0, 1); plt.xticks(rotation=30); plt.ylabel('ICC(1)'); plt.title('Proporción de la varianza explicada por el paciente')
plt.tight_layout(); plt.savefig(f'{RUTA_EDA}/A7_icc.png', dpi=150); plt.show()

---
# Part B. Window level: what the model receives

## B1. Cohort flow

Counts recomputed from `ultimate_dataset_v3_1.csv` and, if available, from the synchronisation report. Used for the flow diagram.

In [ ]:
print("=" * 70); print("B1. FLUJO DE LA COHORTE"); print("=" * 70)
flujo = []
def paso(nombre, ventanas, pacientes, nota=''):
    flujo.append({'Paso': nombre, 'Ventanas': ventanas, 'Pacientes': pacientes, 'Nota': nota})

if reporte is not None:
    paso('Cohorte sincronizada con waveform', None, reporte['icustay_id'].nunique(), 'df_reporte_final_unificado')
paso('Con HRV extraída (≥3 h válidas)', None, hrv.icustay_id.nunique(), 'hrv_completo_reauditado')
paso('Con ≥1 ventana de 12 h', len(ult), ult.icustay_id.nunique(), 'T0 = 12, 18, 24… h')
paso(f'Con ≥{UMBRAL_HRV} h de HRV válida', len(dfv), dfv.icustay_id.nunique(), 'Filtro fijado a priori')
paso('  Muerte en el horizonte', int(dfv.muere_en_horizonte.sum()), dfv.loc[dfv.muere_en_horizonte == 1, 'icustay_id'].nunique(), 'Label forzado')
paso('  Alta viva en el horizonte', int(dfv.alta_viva_en_horizonte.sum()), dfv.loc[dfv.alta_viva_en_horizonte == 1, 'icustay_id'].nunique(), 'Excluidas')
for lab, (nom, _) in LABELS.items():
    s = dfv[dfv[lab].notna()]
    paso(f'Evaluable: {nom}', len(s), s.icustay_id.nunique(), f'prevalencia {s[lab].mean()*100:.1f} %')
s = dfv[dfv.eval_comun == 1]
paso('Cohorte común (SOFA + cardio)', len(s), s.icustay_id.nunique(), 'Sensibilidad')
flujo = pd.DataFrame(flujo); display(flujo); flujo.to_csv(f'{RUTA_EDA}/B1_flujo.csv', index=False)

print("\nSensibilidad del umbral de HRV:")
display(pd.DataFrame([{'umbral': f'≥{u} h', 'ventanas': int(ult[f'incl_hrv_{u}h'].sum()),
                       'pacientes': ult.loc[ult[f'incl_hrv_{u}h'] == 1, 'icustay_id'].nunique()} for u in [4, 6, 8]]))

plt.figure(figsize=(10, 3.5))
sns.countplot(x=ult.n_horas_validas, color='teal')
plt.axvline(UMBRAL_HRV - 0.5, c='orange', ls='--'); plt.text(UMBRAL_HRV - 0.3, plt.ylim()[1] * .9, f'umbral ≥{UMBRAL_HRV} h', color='darkorange')
plt.title('Horas de HRV válidas por ventana candidata'); plt.xlabel('Horas válidas de 12'); plt.ylabel('Ventanas')
plt.tight_layout(); plt.savefig(f'{RUTA_EDA}/B1_horas_por_ventana.png', dpi=150); plt.show()

## B2. Windows per patient

Unweighted, a patient with many windows weighs more in training. This section quantifies that imbalance, which motivates the 1/n_windows weight per patient (computed inside each fold).

In [ ]:
print("=" * 70); print("B2. VENTANAS POR PACIENTE"); print("=" * 70)
filas_b2 = []
fig, axes = plt.subplots(1, 3, figsize=(17, 4))
for ax, (lab, (nom, _)) in zip(axes, LABELS.items()):
    s = dfv[dfv[lab].notna()]
    vpp = s.groupby('icustay_id').size().sort_values(ascending=False)
    top10 = vpp.head(max(1, int(np.ceil(len(vpp) * .10)))).sum() / vpp.sum() * 100
    filas_b2.append({'label': nom, 'pacientes': len(vpp), 'ventanas': vpp.sum(), 'mediana': vpp.median(),
                     'P25': vpp.quantile(.25), 'P75': vpp.quantile(.75), 'max': vpp.max(),
                     '% ventanas del 10 % de pacientes con más ventanas': round(top10, 1)})
    sns.histplot(vpp, discrete=True, ax=ax, color='slategray'); ax.set(title=nom, xlabel='Ventanas por paciente', ylabel='Pacientes')
plt.tight_layout(); plt.savefig(f'{RUTA_EDA}/B2_ventanas_por_paciente.png', dpi=150); plt.show()
tabla_b2 = pd.DataFrame(filas_b2); display(tabla_b2)

## B3. Aggregated HRV features: coverage and distribution

With the >= 6 h filter each window has at least 6 points, so mean, SD, last value and slope should be almost complete. `horas_desde_ultima_hrv` shows how recent the last value is.

In [ ]:
print("=" * 70); print("B3. FEATURES AGREGADAS: NULOS Y DISTRIBUCIÓN"); print("=" * 70)
nulos_feat = analizar_nulos(dfv, FEAT_HRV, f'Features HRV (ventanas ≥{UMBRAL_HRV} h)', graficar=False)

fig, axes = plt.subplots(1, 2, figsize=(14, 4))
sns.countplot(x=dfv.horas_desde_ultima_hrv.astype(int), ax=axes[0], color='teal')
axes[0].set(title='Horas desde la última HRV válida hasta T0', xlabel='Horas', ylabel='Ventanas')
sns.countplot(x=dfv.n_horas_hrv_ventana.astype(int), ax=axes[1], color='slategray')
axes[1].set(title='Horas válidas en la ventana (cohorte analítica)', xlabel='Horas', ylabel='Ventanas')
plt.tight_layout(); plt.show()
print(f"Ventanas cuyo último valor tiene ≥3 h de antigüedad: {(dfv.horas_desde_ultima_hrv >= 3).mean()*100:.1f} %")

desc_feat = dfv[FEAT_HRV].describe().T; desc_feat['asimetria'] = dfv[FEAT_HRV].skew()
display(desc_feat.round(4)); desc_feat.to_csv(f'{RUTA_EDA}/B3_descriptivo_features.csv')

for sufijo, titulo, color in [('mean', 'Media en la ventana', 'teal'), ('sd', 'DE en la ventana', 'slategray'),
                              ('slope', 'Pendiente en la ventana (unidades/hora)', 'coral')]:
    fig, axes = plt.subplots(2, 5, figsize=(22, 7)); axes = axes.flatten()
    for i, m in enumerate(METRICAS):
        c = f'{m}_{sufijo}'; s = dfv[c].dropna()
        if sufijo in ('mean', 'sd') and m in ('LF', 'HF', 'LFHF'):
            s = a_ln(s[s > 0]) if sufijo == 'mean' else s
            axes[i].set_xlabel('ln' if sufijo == 'mean' else 'escala original')
        sns.histplot(s, bins=35, ax=axes[i], color=color); axes[i].set_title(c)
        if sufijo == 'slope': axes[i].axvline(0, c='k', lw=.8)
    plt.suptitle(titulo, y=1.02, fontsize=14); plt.tight_layout(); plt.show()

## B4. Redundancy among the 40 features

For each metric, the mean and the last value are usually highly correlated. Nothing is removed here; this anticipates collinearity and helps read SHAP values.

In [ ]:
print("=" * 70); print("B4. REDUNDANCIA ENTRE FEATURES AGREGADAS"); print("=" * 70)
corr_feat = dfv[FEAT_HRV_40].corr(method='spearman')
cg = sns.clustermap(corr_feat.fillna(0), cmap='coolwarm', center=0, vmin=-1, vmax=1, figsize=(15, 15),
                    xticklabels=True, yticklabels=True, cbar_kws={'label': 'ρ de Spearman'})
cg.ax_heatmap.tick_params(labelsize=8); cg.fig.suptitle('Correlación entre las 40 features HRV (agrupamiento jerárquico)', y=1.01)
plt.savefig(f'{RUTA_EDA}/B4_clustermap.png', dpi=150, bbox_inches='tight'); plt.show()

pares = corr_feat.where(np.triu(np.ones(corr_feat.shape), k=1).astype(bool)).stack().reset_index()
pares.columns = ['var_1', 'var_2', 'rho']; pares = pares.reindex(pares.rho.abs().sort_values(ascending=False).index)
print(f"Pares con |ρ| ≥ 0,9: {(pares.rho.abs() >= .9).sum()} de {len(pares)}")
display(pares[pares.rho.abs() >= .9].round(3).reset_index(drop=True))

print("\nCorrelación media vs último valor, por métrica:")
display(pd.Series({m: dfv[[f'{m}_mean', f'{m}_last']].corr(method='spearman').iloc[0, 1] for m in METRICAS},
                  name='rho(mean, last)').round(3).to_frame())

## B5. Included versus excluded windows

Compared with the SMD (|SMD| < 0.1 negligible; 0.1-0.2 small; > 0.2 relevant):
1. Windows with >= 6 h of HRV versus 1-5 h (windows with 0 h have no HRV).
2. Within the analytic cohort: SOFA label evaluable versus not evaluable.

In [ ]:
print("=" * 70); print("B5. INCLUIDAS vs EXCLUIDAS (SMD)"); print("=" * 70)
def smd(a, b):
    a, b = a.dropna(), b.dropna()
    if len(a) < 2 or len(b) < 2: return np.nan
    sp = np.sqrt((a.var() + b.var()) / 2)
    return (a.mean() - b.mean()) / sp if sp > 0 else np.nan

FEAT_MEAN = [f'{m}_mean' for m in METRICAS]
comp = []
inc, exc = ult[ult.n_horas_validas >= UMBRAL_HRV], ult[(ult.n_horas_validas > 0) & (ult.n_horas_validas < UMBRAL_HRV)]
eva, noeva = dfv[dfv.label_sofa_responder.notna()], dfv[dfv.label_sofa_responder.isna()]
for c in FEAT_MEAN + (['age'] if 'age' in ult.columns else []):
    comp.append({'variable': c,
                 f'SMD ≥{UMBRAL_HRV}h vs 1–{UMBRAL_HRV-1}h': smd(inc[c], exc[c]),
                 'SMD SOFA evaluable vs no': smd(eva[c], noeva[c])})
tabla_b5 = pd.DataFrame(comp).set_index('variable').round(3)
print(f"n: ≥{UMBRAL_HRV}h = {len(inc)} | 1–{UMBRAL_HRV-1}h = {len(exc)} | SOFA evaluable = {len(eva)} | no evaluable = {len(noeva)}")
display(tabla_b5.style.map(lambda v: 'background-color:#f4cccc' if pd.notna(v) and abs(v) > .2 else
                                         ('background-color:#fff2cc' if pd.notna(v) and abs(v) > .1 else '')))
tabla_b5.to_csv(f'{RUTA_EDA}/B5_smd_incluidas_excluidas.csv')

fig, ax = plt.subplots(figsize=(8, 6))
for col, mk in zip(tabla_b5.columns, ['o', 's']):
    ax.scatter(tabla_b5[col], tabla_b5.index, label=col, marker=mk)
for x in [-.2, -.1, .1, .2]: ax.axvline(x, ls='--', c='gray', lw=.7)
ax.axvline(0, c='k', lw=.8); ax.set_xlabel('SMD'); ax.legend(loc='lower right', fontsize=9)
ax.set_title('Diferencias entre ventanas incluidas y excluidas'); plt.tight_layout(); plt.show()

## B6. HRV by label (descriptive)

SMD between classes for the 41 features of each label. **Not feature selection**: windows of one patient are correlated and any selection must happen inside cross-validation.

Positive SMD = higher value in class 1 (SOFA improvement / haemodynamic stability or recovery / progression).

In [ ]:
print("=" * 70); print("B6. HRV POR LABEL (SMD ENTRE CLASES)"); print("=" * 70)
mat_smd = {}
for lab, (nom, _) in LABELS.items():
    s = dfv[dfv[lab].notna()]
    mat_smd[nom] = {c: smd(s.loc[s[lab] == 1, c], s.loc[s[lab] == 0, c]) for c in FEAT_HRV}
    print(f"{nom}: {len(s)} ventanas | clase 1 = {int(s[lab].sum())} | clase 0 = {int((s[lab] == 0).sum())}")
mat_smd = pd.DataFrame(mat_smd); mat_smd.to_csv(f'{RUTA_EDA}/B6_smd_por_label.csv')

plt.figure(figsize=(9, 13))
lim = max(.3, np.nanmax(np.abs(mat_smd.values)))
sns.heatmap(mat_smd, annot=True, fmt='.2f', cmap='coolwarm', center=0, vmin=-lim, vmax=lim, cbar_kws={'label': 'SMD'})
plt.title('Diferencia estandarizada entre clases (clase 1 − clase 0)'); plt.tight_layout()
plt.savefig(f'{RUTA_EDA}/B6_smd_heatmap.png', dpi=150); plt.show()

# Differences in levels or in trends?
tipo = pd.Series({c: c.rsplit('_', 1)[-1] for c in FEAT_HRV_40})
resumen_tipo = mat_smd.loc[FEAT_HRV_40].abs().groupby(tipo).mean().round(3)
print("\n|SMD| medio por tipo de resumen (niveles = mean/last; tendencias = sd/slope):"); display(resumen_tipo)

print("\nTop 10 features por |SMD| en cada label:")
for nom in mat_smd.columns:
    top = mat_smd[nom].abs().sort_values(ascending=False).head(10)
    display(mat_smd.loc[top.index, [nom]].round(3))

In [ ]:
# Boxplots by domain for the main SOFA label (window means)
lab = 'label_sofa_responder'; nom, mapa = LABELS[lab]
s = dfv[dfv[lab].notna()].copy(); s['Clase'] = s[lab].map(mapa)
for dominio, mets in DOMINIOS.items():
    cols = [f'{m}_mean' for m in mets]
    fig, axes = plt.subplots(1, len(cols), figsize=(4.2 * len(cols), 4))
    axes = np.atleast_1d(axes)
    for ax, c in zip(axes, cols):
        sns.boxplot(data=s, x='Clase', y=c, ax=ax, palette='Set2', fliersize=1.5)
        if c.split('_')[0] in ('LF', 'HF', 'LFHF'): ax.set_yscale('log')
        ax.set_title(c); ax.set_xlabel('')
    plt.suptitle(f'{dominio}: {nom}', y=1.03, fontsize=13); plt.tight_layout(); plt.show()

In [ ]:
# By sex and type of sepsis onset (if available)
lab = 'label_sofa_responder'; nom, mapa = LABELS[lab]
s = dfv[dfv[lab].notna()].copy(); s['Clase'] = s[lab].map(mapa)
segmentos = []
if 'gender' in s.columns: segmentos.append('gender')
if reporte is not None and 'tipo_t0' in reporte.columns:
    s = s.merge(reporte[['icustay_id', 'tipo_t0']].drop_duplicates('icustay_id'), on='icustay_id', how='left')
    segmentos.append('tipo_t0')
    print("Ventanas por tipo de inicio de sepsis:"); display(s.tipo_t0.value_counts(dropna=False))

cols = ['RMSSD_mean', 'SDNN_mean', 'DFA_a1_mean', 'SampEn_mean']
for seg in segmentos:
    fig, axes = plt.subplots(1, len(cols), figsize=(5 * len(cols), 4))
    for ax, c in zip(axes, cols):
        sns.boxplot(data=s, x=seg, y=c, hue='Clase', ax=ax, palette='Set2', fliersize=1.5)
        ax.set_title(c); ax.tick_params(axis='x', rotation=15)
    plt.suptitle(f'{nom} segmentado por {seg}', y=1.03, fontsize=13); plt.tight_layout(); plt.show()

## B7. Trajectories over time

Mean of selected metrics by T0 hour, split by SOFA-label class. Each patient contributes at most one window per T0 hour, so each 95 % CI is between patients; points at different hours share patients, so the curve is a visual guide, not a test.

In [ ]:
print("=" * 70); print("B7. TRAYECTORIAS SEGÚN LA HORA DEL T0"); print("=" * 70)
lab = 'label_sofa_responder'; nom, mapa = LABELS[lab]
s = dfv[dfv[lab].notna()].copy(); s['Clase'] = s[lab].map(mapa)
print("Ventanas por hora del T0 y clase:"); display(pd.crosstab(s.hour_index, s.Clase).T)

cols = ['SDNN_mean', 'RMSSD_mean', 'LFHF_mean', 'SampEn_mean', 'DFA_a1_mean']
d = s.melt(id_vars=['hour_index', 'Clase'], value_vars=cols, var_name='Métrica', value_name='Valor')
g = sns.relplot(data=d, x='hour_index', y='Valor', hue='Clase', col='Métrica', col_wrap=3, kind='line',
                marker='o', errorbar=('ci', 95), facet_kws={'sharey': False}, height=3.4, aspect=1.3, palette='Set2')
g.set_axis_labels('Hora del T0 desde el inicio de la sepsis', 'Media en la ventana')
g.figure.suptitle(f'Trayectoria de HRV por clase — {nom}', y=1.03)
plt.savefig(f'{RUTA_EDA}/B7_trayectorias.png', dpi=150, bbox_inches='tight'); plt.show()

## B8. Trajectory overlap by domain

Rows are the three labels; each figure groups one domain (time, frequency, non-linear). Also counts the T0 hours in which the 95 % CIs of the two classes do not overlap. Points at different hours share patients: the curve is not a set of independent tests.

In [ ]:
# B8. Trajectory overlap by domain (time, frequency, non-linear).
# Rows = the 3 labels; columns = metrics of the domain; colour = class; x = T0 hour.
# One window per patient per T0 hour, so each 95 % CI is between patients;
# points at different hours share patients (not independent tests).
from scipy import stats

RESUMEN = 'mean'  # 'mean', 'last', 'sd' or 'slope'
MIN_POR_CLASE = 5  # T0 hours with fewer windows per class are skipped (unstable CIs)
METRICAS_LOG = {'LF', 'HF', 'LFHF'}  # plotted on the ln scale (a_ln detects the scale)
CLASE_1, CLASE_0 = 'Responder / deterioro (1)', 'No responder / sin deterioro (0)'

print("=" * 70); print(f"B8. SOBRELAPAMIENTO POR DOMINIO — resumen de ventana: {RESUMEN}"); print("=" * 70)

def nombre_col(m):
    return f'ln {m}_{RESUMEN}' if (m in METRICAS_LOG and RESUMEN in ('mean', 'last')) else f'{m}_{RESUMEN}'

# 1. Long format: one row per (window, evaluable label, metric)
bloques = []
for lab, (nom, _) in LABELS.items():
    s = dfv[dfv[lab].notna()].copy()
    # T0 hours with enough windows in both classes
    ct = pd.crosstab(s.hour_index, s[lab])
    horas_ok = ct[(ct.get(0.0, 0) >= MIN_POR_CLASE) & (ct.get(1.0, 0) >= MIN_POR_CLASE)].index
    s = s[s.hour_index.isin(horas_ok)]
    for m in METRICAS:
        v = s[f'{m}_{RESUMEN}']
        if m in METRICAS_LOG and RESUMEN in ('mean', 'last'):
            v = a_ln(v.where(v > 0)).values
        bloques.append(pd.DataFrame({'icustay_id': s.icustay_id, 'hour_index': s.hour_index,
                                     'Label': nom, 'Clase': np.where(s[lab] == 1, CLASE_1, CLASE_0),
                                     'Métrica': nombre_col(m), 'Valor': np.asarray(v)}))
    print(f"{nom}: horas del T0 con ≥{MIN_POR_CLASE} ventanas por clase → {list(horas_ok)}")
largo = pd.concat(bloques, ignore_index=True).dropna(subset=['Valor'])
ORDEN_LABELS = [LABELS[l][0] for l in LABELS]

# 2. Plots by domain
def graficar_dominio(dominio, metricas):
    cols = [nombre_col(m) for m in metricas]
    d = largo[largo['Métrica'].isin(cols)]
    if d.empty:
        print(f"Sin datos para {dominio}"); return
    g = sns.relplot(data=d, x='hour_index', y='Valor', hue='Clase', hue_order=[CLASE_0, CLASE_1],
                    row='Label', row_order=[r for r in ORDEN_LABELS if r in d.Label.unique()],
                    col='Métrica', col_order=cols, kind='line', marker='o', dashes=False,
                    errorbar=('ci', 95), palette=['#4c72b0', '#c44e52'],
                    facet_kws={'sharey': 'col', 'sharex': True}, height=3.2, aspect=1.25)
    g.set_titles('{row_name}\n{col_name}', size=10)
    g.figure.subplots_adjust(hspace=0.45)
    g.set_axis_labels('Hora del T0 desde el inicio de la sepsis', f'{RESUMEN} (IC 95 %)')
    g.figure.suptitle(f'Sobrelapamiento de trayectorias — dominio {dominio.lower()}', y=1.07, fontsize=15, fontweight='bold')
    sns.move_legend(g, 'lower center', bbox_to_anchor=(.5, 1.0), ncol=2, title=None, frameon=False)
    plt.savefig(f'{RUTA_EDA}/B8_sobrelapamiento_{dominio.lower().replace(" ", "_")}_{RESUMEN}.png', dpi=150, bbox_inches='tight')
    plt.show()

for dominio, mets in DOMINIOS.items():
    graficar_dominio(dominio, mets)

# 3. Overlap: T0 hours in which the 95 % CIs do not touch
def ic95(x):
    x = x.dropna(); n = len(x)
    if n < 2: return np.nan, np.nan
    h = stats.t.ppf(.975, n - 1) * x.std(ddof=1) / np.sqrt(n)
    return x.mean() - h, x.mean() + h

filas = []
for (lab, met, hora), grp in largo.groupby(['Label', 'Métrica', 'hour_index']):
    lo1, hi1 = ic95(grp.loc[grp.Clase == CLASE_1, 'Valor'])
    lo0, hi0 = ic95(grp.loc[grp.Clase == CLASE_0, 'Valor'])
    if np.isnan([lo1, hi1, lo0, hi0]).any(): continue
    sin_solape = (hi1 < lo0) or (hi0 < lo1)
    filas.append({'Label': lab, 'Métrica': met, 'hora': hora, 'sin_solape': sin_solape,
                  'dirección': ('clase 1 mayor' if lo1 > hi0 else 'clase 1 menor') if sin_solape else ''})
solape = pd.DataFrame(filas)
resumen_solape = (solape.groupby(['Métrica', 'Label'])
                  .agg(horas_evaluadas=('sin_solape', 'size'), horas_sin_solape=('sin_solape', 'sum'))
                  .assign(pct_sin_solape=lambda t: (100 * t.horas_sin_solape / t.horas_evaluadas).round(1))
                  .reset_index())
resumen_solape.to_csv(f'{RUTA_EDA}/B8_solape_{RESUMEN}.csv', index=False)

piv = resumen_solape.pivot(index='Métrica', columns='Label', values='pct_sin_solape') \
                    .reindex(index=[nombre_col(m) for m in METRICAS], columns=[c for c in ORDEN_LABELS if c in resumen_solape.Label.unique()])
plt.figure(figsize=(8, 6))
sns.heatmap(piv, annot=True, fmt='.0f', cmap='Reds', vmin=0, vmax=100, cbar_kws={'label': '% de horas sin solape'})
plt.title(f'% de horas del T0 en que los IC 95 % de las clases no se solapan ({RESUMEN})')
plt.xlabel(''); plt.ylabel(''); plt.tight_layout()
plt.savefig(f'{RUTA_EDA}/B8_solape_heatmap_{RESUMEN}.png', dpi=150); plt.show()

print("\nHoras sin solape por métrica y label:"); display(piv)
det = solape[solape.sin_solape].sort_values(['Label', 'Métrica', 'hora'])
if len(det):
    print("\nDetalle de las horas sin solape:"); display(det[['Label', 'Métrica', 'hora', 'dirección']].reset_index(drop=True))

print('''
>> CÓMO LEERLO
   - Que los IC 95 % no se toquen es un criterio exigente (equivale aprox. a p < 0,01);
     que se toquen NO significa que no haya diferencia.
   - Con muchas horas y métricas, algún punto sin solape aparece por azar. Una señal
     consistente es la que se repite en varias horas seguidas y en la misma dirección.
   - Es descriptivo: no se usa para elegir features. Cambia RESUMEN a 'slope' o 'sd'
     para ver si la separación está en las tendencias y no en los niveles.
''')

---
## Reading guide

| Section | What to look at | Used for |
|---|---|---|
| A2 | How coverage falls over time | Limitation: late windows represent patients with prolonged monitoring |
| A5 | Critical violations | Would be fixed in the extraction, not here |
| A6, B4 | Redundant pairs (expected: SD1 ~ RMSSD) | Interpretation of SHAP and coefficients |
| A7 | Stable versus fluctuating metrics | Levels versus trends |
| B2 | Concentration of windows per patient | Justifies the 1/n weight inside the fold |
| B5 | SMD included versus excluded | Selection bias to report |
| B6, B8 | Differences in levels or trends | Context only; **not** feature selection |

This notebook does not modify the dataset. Tables and figures are written to `eda_hrv_v3_1/`.

In [ ]:
print("Archivos generados en", RUTA_EDA)
for f in sorted(os.listdir(RUTA_EDA)):
    print("  -", f)